# Youtube Phrase (N-grams) Extraction Tool
This notebook contains the implementation for extracting transcripts and analyzing frequent phrases in Youtube videos.

In [ ]:
!pip install youtube-transcript-api nltk

In [ ]:
import json
import queue
import re
import concurrent.futures
from youtube_transcript_api import YouTubeTranscriptApi
from collections import defaultdict, Counter
import nltk
from nltk.corpus import stopwords
import string

# Download the stopwords if they are not already available
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish')) # Assuming Spanish, but it can be modified


## Stage 1: Transcripts Download (In Parallel)

In this stage, transcripts are obtained from the provided URLs using `youtube-transcript-api` and saved in temporary JSON files. The identifiers of these files are queued for later processing.

In [ ]:
# Native Python queue to pass information to stage 2
processing_queue = queue.Queue()

# 1. Instanciamos el API una sola vez para ser usada globalmente o dentro de las funciones
ytt_api = YouTubeTranscriptApi()

# Native Python queue to pass information to stage 2
processing_queue = queue.Queue()

def extract_video_id(url):
    """Extrae el ID del video de YouTube desde la URL."""
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11}).*', url)
    return match.group(1) if match else None

def process_video(url, creator):
    """Downloads the video transcript and saves it in a temporary JSON."""
    video_id = extract_video_id(url)
    if not video_id:
        print(f"Error: No se pudo extraer el ID de la URL: {url}")
        return

    try:
        # MODIFICATION: Use of the instance and fetch() method according to documentation
        # languages=['es', 'en'] tries Spanish first, then English
        fetched_transcript = ytt_api.fetch(video_id, languages=['es', 'en'])

        # Convertimos el objeto FetchedTranscript a una lista de diccionarios (raw data)
        transcript = fetched_transcript.to_raw_data()

        # Unique identifier for the file
        identifier = f"{creator.replace(' ', '_')}_{video_id}"
        output_file = f"{identifier}.json"

        # Guardar en JSON temporal
        data = {
            "creator": creator,
            "url": url,
            "video_id": video_id,
            "transcript": transcript
        }

        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"Success: Transcript of '{creator}' saved in {output_file}")

        # Enviar identifier a la cola
        processing_queue.put(identifier)

    except Exception as e:
        # If the video does not have subtitles in those languages, it will enter here
        print(f"Error processing {url} (Creator: {creator}): {str(e)}")

def execute_stage_1(url_dictionary, num_threads=4):
    """Ejecuta la descarga de transcripciones en paralelo."""
    print(f"Starting Stage 1 with {num_threads} threads...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Send tasks al pool de hilos
        futures = [executor.submit(process_video, url, creator) for url, creator in url_dictionary.items()]

        # Esperar a que terminen
        concurrent.futures.wait(futures)

    print(f"Stage 1 finished. Files in queue: {processing_queue.qsize()}")

# Ejemplo de uso
test_dictionary = {
    "https://www.youtube.com/watch?v=mJcJ0ikucOY": "XOCAS",
    "https://www.youtube.com/watch?v=aMcTiuqqJas": "GUSGRI",
    "https://www.youtube.com/watch?v=kewy2RUYb3c": "XOCAS",
    "https://www.youtube.com/watch?v=n9RhsFlKeSI": "XOCAS",
    "https://www.youtube.com/watch?v=S8q9eHBnHxE": "XOCAS",
    "https://www.youtube.com/watch?v=dnWT6lV8Zms": "GUSGRI",
    "https://www.youtube.com/watch?v=yUAbe5isKTg": "GUSGRI",
    "https://www.youtube.com/watch?v=5rF-0wK4fpI": "GUSGRI",
    "https://www.youtube.com/watch?v=R2BYWHmhBuk": "GUSGRI",
    "https://www.youtube.com/watch?v=8V6b5hCx5F8": "XOCAS",
    "https://www.youtube.com/watch?v=tf4LAo_JY68": "XOCAS",
    "https://www.youtube.com/watch?v=Bfy1yDB_YXM": "XOCAS",
    "https://www.youtube.com/watch?v=JaqdFMzWmqU": "XOCAS",
    "https://www.youtube.com/watch?v=tUsXRMXlHl0": "GUSGRI",
    "https://www.youtube.com/watch?v=PYPsdJmcgjo": "GUSGRI",
    "https://www.youtube.com/watch?v=i49LvJMtwbM": "GUSGRI",
    "https://www.youtube.com/watch?v=p5jo4kS2P8Q": "GUSGRI",
    "https://www.youtube.com/watch?v=fQxVMOSB_L8": "GUSGRI",
    "https://www.youtube.com/watch?v=SOnBk94vOzk": "GUSGRI",
    "https://www.youtube.com/watch?v=-ibeiAm_VEY": "GUSGRI"
}

# Execute Stage 1
execute_stage_1(test_dictionary, num_threads=2)


## Stage 2: N-grams Processing (In Parallel)

In this stage, the queued files from Stage 1 are processed. The most used n-grams by each creator are extracted (from 2 to 6 words), optionally discarding those that are composed 100% of stop words. The timestamp of the first appearance of the n-gram is captured.

In [ ]:
def clean_word(word):
    """Converts to lowercase and removes punctuation signs."""
    word = word.lower()
    return word.translate(str.maketrans('', '', string.punctuation))

def is_valid_ngram(ngram_words, use_stopwords_filter=True):
    """
    Business rule: If the filter is active, the n-gram is discarded
    SOLO si TODAS las words que lo componen son stopwords.
    """
    if not use_stopwords_filter:
        return True

    # Validate if all words are in the stopwords list
    are_all_stopwords = all(word in stop_words for word in ngram_words)

    # It is valid if NOT all are stopwords (i.e., it has at least one valuable word)
    return not are_all_stopwords

def tokenize_transcript(transcript):
    """
    Decomposes the transcript into a list of words where each word
    retiene el inicio (start) y el fin (start + duration) de su block original.
    """
    tokens = []
    for block in transcript:
        text = block.get('text', '')
        start = block.get('start', 0)
        end = start + block.get('duration', 0)

        words = text.split()
        for p in words:
            clean_p = clean_word(p)
            if clean_p: # Avoid empty strings
                tokens.append({
                    'word': clean_p,
                    'start': start,
                    'end': end
                })
    return tokens

def extract_video_ngrams(identifier, n_phrases, use_stopwords_filter=True):
    """Processes a JSON file, generates n-grams from 2 to 6 words and returns the results."""
    file_name = f"{identifier}.json"

    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            data = json.load(f)

        transcript = data['transcript']
        creator = data['creator']
        url = data['url']

        tokens = tokenize_transcript(transcript)

        # Dictionary to store counts and first appearance info
        # Estructura: { n_size: { "ngrama_str": {"count": int, "start": float, "end": float, "url": str} } }
        video_results = {i: {} for i in range(2, 7)}

        total_tokens = len(tokens)

        for n in range(2, 7): # N-grams of size 2, 3, 4, 5, 6
            for i in range(total_tokens - n + 1):
                ngram_tokens = tokens[i:i+n]
                ngram_words = tuple(t['word'] for t in ngram_tokens)

                # Apply business rule (discard if 100% stopwords)
                if not is_valid_ngram(ngram_words, use_stopwords_filter):
                    continue

                ngram_str = " ".join(ngram_words)

                # The start is the first token, the end is the last token
                start_time = ngram_tokens[0]['start']
                end_time = ngram_tokens[-1]['end']

                appearance = {'start': start_time, 'end': end_time, 'url': url}
                if ngram_str in video_results[n]:
                    video_results[n][ngram_str]['count'] += 1
                    video_results[n][ngram_str]['appearances'].append(appearance)
                else:
                    video_results[n][ngram_str] = {
                        'count': 1,
                        'start': start_time,
                        'end': end_time,
                        'url': url,
                        'appearances': [appearance]
                    }

        return creator, video_results

    except Exception as e:
        print(f"Error processing file {file_name}: {str(e)}")
        return None, None

def merge_results(global_results, creator, video_results):
    """Sums the counts of the n-grams of a video to the creator's totals."""
    if creator not in global_results:
        global_results[creator] = {i: {} for i in range(2, 7)}

    for n in range(2, 7):
        for ngram_str, info in video_results[n].items():
            if ngram_str in global_results[creator][n]:
                global_results[creator][n][ngram_str]['count'] += info['count']
                # Keep the start/end/url of the first appearance (do not overwrite)
                # Extend the appearances list
                if 'appearances' in info:
                    global_results[creator][n][ngram_str]['appearances'].extend(info['appearances'])
            else:
                global_results[creator][n][ngram_str] = info

def execute_stage_2(n_phrases=10, num_threads=4, use_stopwords_filter=True):
    """Empties the queue, processes in parallel, and extracts the Top N phrases."""
    items_to_process = []

    # Empty the queue
    while not processing_queue.empty():
        items_to_process.append(processing_queue.get())

    if not items_to_process:
        print("The queue is empty. Execute Stage 1 first.")
        return

    print(f"Starting Stage 2: Processing {len(items_to_process)} videos with {num_threads} threads...")

    global_results = {} # { creator: { n: { ngrama: info } } }

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Send tasks
        futures = [executor.submit(extract_video_ngrams, ident, n_phrases, use_stopwords_filter) for ident in items_to_process]

        # Collect results as they finish
        for futuro in concurrent.futures.as_completed(futures):
            creator, video_results = futuro.result()
            if creator and video_results:
                merge_results(global_results, creator, video_results)

    # Extract the Top N per creator
    top_results = {}
    for creator, ngrams_by_size in global_results.items():
        top_results[creator] = {}
        for n in range(2, 7):
            # Sort by count (descending)
            sorted_items = sorted(ngrams_by_size[n].items(), key=lambda x: x[1]['count'], reverse=True)
            # Take the top N
            top_results[creator][n] = sorted_items[:n_phrases]

    print("Stage 2 finished.")
    return top_results

def show_results(top_results):
    """Formats and displays the results obtained in two parts and saves the JSONs."""
    if not top_results:
        return

    json_first = {}
    json_all = {}

    print("\n" + "*"*60)
    print("RESULTS: FIRST APPEARANCE")
    print("*"*60)
    for creator, results in top_results.items():
        json_first[creator] = {}
        print(f"\n{'='*50}")
        print(f"CREATOR: {creator}")
        print(f"{'='*50}")

        for n in range(2, 7):
            json_first[creator][f"{n}_grams"] = []
            print(f"\n--- Top N-grams of {n} words ---")
            ngrams = results.get(n, [])
            if not ngrams:
                print("No results found.")
                continue

            for i, (phrase, info) in enumerate(ngrams, 1):
                count = info['count']
                start = info['start']
                end = info['end']
                url = info['url']

                json_first[creator][f"{n}_grams"].append({
                    'phrase': phrase,
                    'repetitions': count,
                    'first_appearance': {'start': start, 'end': end, 'url': url}
                })

                print(f"{i}. '{phrase}' (Repetitions: {count})")
                print(f"   First appearance: {start:.2f}s - {end:.2f}s | Source: {url}")

    print("\n" + "*"*60)
    print("RESULTS: ALL APPEARANCES")
    print("*"*60)
    for creator, results in top_results.items():
        json_all[creator] = {}
        print(f"\n{'='*50}")
        print(f"CREATOR: {creator}")
        print(f"{'='*50}")

        for n in range(2, 7):
            json_all[creator][f"{n}_grams"] = []
            print(f"\n--- Top N-grams of {n} words ---")
            ngrams = results.get(n, [])
            if not ngrams:
                print("No results found.")
                continue

            for i, (phrase, info) in enumerate(ngrams, 1):
                count = info['count']
                appearances = info.get('appearances', [])

                json_all[creator][f"{n}_grams"].append({
                    'phrase': phrase,
                    'repetitions': count,
                    'appearances': appearances
                })

                print(f"{i}. '{phrase}' (Repetitions: {count})")
                for j, ap in enumerate(appearances, 1):
                    print(f"   Appearance {j}: {ap['start']:.2f}s - {ap['end']:.2f}s | Source: {ap['url']}")

    with open('first_appearance_result.json', 'w', encoding='utf-8') as f:
        json.dump(json_first, f, ensure_ascii=False, indent=2)
    print("\nFile 'first_appearance_result.json' generated successfully.")

    with open('all_appearances_result.json', 'w', encoding='utf-8') as f:
        json.dump(json_all, f, ensure_ascii=False, indent=2)
    print("File 'all_appearances_result.json' generated successfully.")

# Execute Stage 2 and show results
results = execute_stage_2(n_phrases=10, num_threads=2, use_stopwords_filter=True)
show_results(results)
